In [1]:
import numpy as np
import random

def is_win(board, player):
    """Check if the player has won."""
    for row in range(3):
        if all(board[row, col] == player for col in range(3)):
            return True
    for col in range(3):
        if all(board[row, col] == player for row in range(3)):
            return True
    if all(board[i, i] == player for i in range(3)) or all(board[i, 2 - i] == player for i in range(3)):
        return True
    return False

def is_draw(board):
    """Check if the game is a draw."""
    return np.all(board != 0)

def generate_data(num_games=10000):
    """Generate training data."""
    X, y = [], []
    for _ in range(num_games):
        board = np.zeros((3, 3), dtype=int)
        player = 1  # Player 1 starts
        moves = []

        while True:
            empty_cells = [(i, j) for i in range(3) for j in range(3) if board[i, j] == 0]
            if not empty_cells:
                break

            move = random.choice(empty_cells)
            board[move] = player
            moves.append((board.copy(), move, player))

            if is_win(board, player):
                for state, mv, pl in moves:
                    X.append(state.flatten())
                    y.append(mv[0] * 3 + mv[1])
                break

            if is_draw(board):
                break

            player = 3 - player  # Switch players (1 ↔ 2)

    return np.array(X), np.array(y)

# Generate training data
X, y = generate_data()

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Preprocess data
X = X / 2  # Normalize (board values: 0, 1, 2 → 0, 0.5, 1)
y = to_categorical(y, num_classes=9)

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Build the model
model = Sequential([
    Flatten(input_shape=(9,)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(9, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=32)

def ai_move(board, model):
    """AI makes a move using the trained model."""
    input_board = board.flatten() / 2  # Normalize
    probs = model.predict(input_board.reshape(1, -1))[0]

    # Find the best valid move
    valid_moves = [(i, j) for i in range(3) for j in range(3) if board[i, j] == 0]
    best_move = max(valid_moves, key=lambda mv: probs[mv[0] * 3 + mv[1]])
    return best_move

def print_board(board):
    symbols = {0: ".", 1: "X", 2: "O"}
    for row in board:
        print(" ".join(symbols[cell] for cell in row))

def play_game(model):
    board = np.zeros((3, 3), dtype=int)
    player_turn = True

    while True:
        print_board(board)

        if is_win(board, 1):
            print("Player wins!")
            break
        elif is_win(board, 2):
            print("AI wins!")
            break
        elif is_draw(board):
            print("It's a draw!")
            break

        if player_turn:
            row = int(input("Enter row (0-2): "))
            col = int(input("Enter col (0-2): "))
            if board[row, col] == 0:
                board[row, col] = 1
                player_turn = False
            else:
                print("Invalid move! Try again.")
        else:
            print("AI is thinking...")
            move = ai_move(board, model)
            board[move] = 2
            player_turn = True

# Play the game
play_game(model)


/usr/local/lib/python3.10/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1633/1633 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.3791 - loss: 1.6686 - val_accuracy: 0.3979 - val_loss: 1.3511
Epoch 2/10
1633/1633 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4018 - loss: 1.3232 - val_accuracy: 0.4144 - val_loss: 1.2613
Epoch 3/10
1633/1633 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.4167 - loss: 1.2502 - val_accuracy: 0.4368 - val_loss: 1.2303
Epoch 4/10
1633/1633 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4482 - loss: 1.2091 - val_accuracy: 0.4506 - val_loss: 1.2063
Epoch 5/10
1633/1633 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.4686 - loss: 1.1808 - val_accuracy: 0.4682 - val_loss: 1.1797
Epoch 6/10
1633/1633 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4822 - loss: 1.1575 - val_accuracy: 0.4731 - val_loss: 1.1714
Epoch 7/10
1633/1633 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.4954 - loss: 1.1433 - val_accuracy: 0.4883 - val_loss: 1.1524
Epoch 8/10
1633/1633 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5008 - loss: 1.1293 - 